In [10]:
!pip install pymc arviz --quiet
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


In [11]:
from google.colab import drive
drive.mount('/content/drive')
tmp_dir = "/content/drive/MyDrive/"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
df = pd.read_csv("/content/drive/MyDrive/gemma12b_SimpleQA_experiment_data.csv")

In [13]:
def find_nan_in_df(df):
    if df.isna().any().any():
        nan_columns = df.columns[df.isna().any()].tolist()
        nan_counts = df.isna().sum()
        total_nans = df.isna().sum().sum()
        rows_with_nan = df[df.isna().any(axis=1)]
        num_rows_with_nan = len(rows_with_nan)
        df_no_nans = df.dropna()
    return df_no_nans


def get_first_turn_confidence_initial_chosen(row):
    if row['initial_answer'] == '1':
        return row['confidence_1']
    elif row['initial_answer'] == '2':
        return row['confidence_2']
    elif row['initial_answer'] == '3':
        return row['confidence_3']
    elif row['initial_answer'] == '4':
        return row['confidence_4']
    else:
        return np.nan

def get_second_turn_confidence_initial_chosen(row):
    if row['initial_answer'] == '1':
        return row['second_turn_confidence_1']
    elif row['initial_answer'] == '2':
        return row['second_turn_confidence_2']
    elif row['initial_answer'] == '3':
        return row['second_turn_confidence_3']
    elif row['initial_answer'] == '4':
        return row['second_turn_confidence_4']
    else:
        return np.nan

def get_first_turn_confidence_final_chosen(row):
    if row['final_answer'] == '1':
        return row['confidence_1']
    elif row['final_answer'] == '2':
        return row['confidence_2']
    elif row['final_answer'] == '3':
        return row['confidence_3']
    elif row['final_answer'] == '4':
        return row['confidence_4']
    else:
        return np.nan

def get_second_turn_confidence_final_chosen(row):
    if row['final_answer'] == '1':
        return row['second_turn_confidence_1']
    elif row['final_answer'] == '2':
        return row['second_turn_confidence_2']
    elif row['final_answer'] == '3':
        return row['second_turn_confidence_3']
    elif row['final_answer'] == '4':
        return row['second_turn_confidence_4']
    else:
        return np.nan

df['second_turn_confidence_final_chosen'] = df.apply(get_second_turn_confidence_final_chosen, axis=1)

def create_advice_direction_column(df):
    direction_mapping = {
        "Same": 1,
        "Opposite": -1,
        "Nothing": 0
    }
    df['advice_direction'] = df['other_llm_answer_type'].map(direction_mapping)
    return df

def create_confirmation_flag_column(df):
    direction_mapping = {
        "Same": 1,
        "Opposite": 0,
        "Nothing": 0
    }
    df['confirmation_flag'] = df['other_llm_answer_type'].map(direction_mapping)
    return df

def create_shown_flag_column(df):
    shown_mapping = {
        "shown": 1,
        "hidden": 0
    }
    df['shown_flag'] = df['initial_answer_display'].map(shown_mapping)
    return df

In [14]:
df_no_nans = find_nan_in_df(df)
df = df_no_nans

invalid_initial_answers = df[~df['initial_answer'].isin(['1', '2', '3', '4'])]

df['first_turn_confidence_initial_chosen'] = df.apply(get_first_turn_confidence_initial_chosen, axis=1)
df['second_turn_confidence_initial_chosen'] = df.apply(get_second_turn_confidence_initial_chosen, axis=1)
df['first_turn_confidence_final_chosen'] = df.apply(get_first_turn_confidence_final_chosen, axis=1)
df['second_turn_binary_correctness'] = (df['final_answer'] == df['gt_answer']).astype(int)

df = create_advice_direction_column(df)
df = create_confirmation_flag_column(df)
df = create_shown_flag_column(df)
df["final_answer"] = df["final_answer"].astype(int)
df["initial_answer"] = df["initial_answer"].astype(int)

df.rename(columns={'other_LLM_answer': 'other_llm_answer'}, inplace=True)
df["other_llm_answer"] = df["other_llm_answer"].apply(
    lambda x: -1 if isinstance(x, str) and "hidden from" in x else x
)

df["other_llm_answer"] = pd.to_numeric(df["other_llm_answer"], errors="coerce")
df['other_llm_accuracy'] = df['other_llm_accuracy'] / 100

epsilon_acc = 1e-9
df["other_llm_accuracy_capped"] = np.clip(df["other_llm_accuracy"], a_min=epsilon_acc, a_max=1 - epsilon_acc)

df['effective_other_llm_accuracy_capped'] = df.apply(
    lambda row: row['other_llm_accuracy_capped'] if row['other_llm_answer_type'] in ["Same", "Nothing"] else 1 - row['other_llm_accuracy_capped'],
    axis=1
)

df['advice_direction_final'] = np.where(
    np.abs(df['advice_direction']) < 0.001, 0,
    np.where(df['other_llm_answer'] == df['final_answer'], 1, -1)
)

In [15]:
df["condition_label"] = df["other_llm_answer_type"] + "_" + df["initial_answer_display"]

condition_labels = sorted(df["condition_label"].unique())

label_to_index = {label: i for i, label in enumerate(condition_labels)}

df["condition_idx"] = df["condition_label"].map(label_to_index)

df["advice_type_code"] = df["other_llm_answer_type"].map({"Opposite": 0, "Same": 1, "Nothing": 2})
df["display_type_code"] = df["initial_answer_display"].map({"shown": 0, "hidden": 1})

In [16]:
num_questions = 10000
df_train = df.sample(n=num_questions, random_state=42)

df_remaining = df.drop(df_train.index)
remaining_trials = len(df_remaining)

n_q = 10000
df_test = df_remaining.sample(n=n_q, random_state=123)
df = df_test

In [17]:
def build_latent_model_priorsplit(df, n_conditions = len(condition_labels)):
    n_options = 4
    baseline = 0.25
    rescale = True
    with pm.Model() as model:
        prior_conf_initial = pm.Data("first_turn_confidence_initial_chosen", df["first_turn_confidence_initial_chosen"].values)
        prior_conf_final = pm.Data("first_turn_confidence_final_chosen", df["first_turn_confidence_final_chosen"].values)
        advice_accuracy = pm.Data("other_llm_accuracy_capped", df["other_llm_accuracy_capped"].values)
        advice_direction = pm.Data("advice_direction", df["advice_direction"].values)
        other_llm_answer = pm.Data("other_llm_answer", df["other_llm_answer"].values)
        final_answer = pm.Data("final_answer", df["final_answer"].values)
        initial_answer = pm.Data("initial_answer", df["initial_answer"].values)
        initial_correct = pm.Data("initial_binary_correctness", df["initial_binary_correctness"].values)
        switch_flag = pm.Data("switch_flag", df["change_of_mind"].values)
        shown_flag = pm.Data("shown_flag", df["shown_flag"].values)
        advice_type = pm.Data("advice_type_code", df["advice_type_code"].values)
        display_type = pm.Data("display_type_code", df["display_type_code"].values)

        switch_obs_data = pm.Data("switch_obs_data", df["change_of_mind"].to_numpy())
        final_confidence_final_chosen_obs_data = pm.Data("final_confidence_final_chosen_obs_data",
                                                    np.clip(df["second_turn_confidence_final_chosen"].to_numpy(), 1e-4, 1-1e-4))
        final_confidence_initial_chosen_obs_data = pm.Data("final_confidence_initial_chosen_obs_data",
                                                      np.clip(df["second_turn_confidence_initial_chosen"].to_numpy(), 1e-4, 1-1e-4))

        prior_conf_effective_final  = pm.math.clip(prior_conf_final, 1e-6, 1-1e-6)
        prior_conf_effective_initial  = pm.math.clip(prior_conf_initial, 1e-6, 1-1e-6)

        def effective_advice_prob(advice_direction, advice_for_choice, advice_accuracy, n_options):
            return pm.math.switch(
                pm.math.eq(advice_direction, 0),
                1.0 / n_options,
                pm.math.switch(
                    advice_for_choice,
                    advice_accuracy,
                    (1 - advice_accuracy) / (n_options - 1)
                )
            )

        advice_for_initial_choice = pm.math.eq(other_llm_answer, initial_answer)

        effective_advice_prob_rel_initial = effective_advice_prob(
            advice_direction,
            advice_for_initial_choice,
            advice_accuracy,
            n_options
        )

        advice_for_final_choice = pm.math.eq(other_llm_answer, final_answer)

        effective_advice_prob_rel_final = effective_advice_prob(
            advice_direction,
            advice_for_final_choice,
            advice_accuracy,
            n_options
        )

        effective_advice_prob_initial = effective_advice_prob_rel_initial
        effective_advice_prob_final = effective_advice_prob_rel_final

        advice_direction_final = pm.math.switch(
            pm.math.eq(advice_direction, 0),
            0,
            pm.math.switch(
                pm.math.eq(other_llm_answer, final_answer),
                1,
                -1
            )
        )

        if rescale:
            effective_advice_prob_initial_rescaled = pm.math.switch(
                pm.math.eq(advice_direction, 1),
                (effective_advice_prob_initial - 0.25) / 0.75,
                pm.math.switch(
                    pm.math.eq(advice_direction, -1),
                    (0.25 - effective_advice_prob_initial) / 0.25,
                    0
                )
            )

            effective_advice_prob_final_rescaled = pm.math.switch(
                pm.math.eq(advice_direction_final, 1),
                (effective_advice_prob_final - 0.25) / 0.75,
                pm.math.switch(
                    pm.math.eq(advice_direction_final, -1),
                    (0.25 - effective_advice_prob_final) / 0.25,
                    0
                )
            )

        intercept_initial_conf = pm.Normal("intercept_initial_conf", mu=-1.1, sigma=1)
        intercept_final_conf = pm.Normal("intercept_final_conf", mu=-1.1, sigma=1)
        intercept_switch = pm.Normal("intercept_switch", mu=0, sigma=1)

        w_shown_shared = pm.Normal("w_shown_shared", mu=0, sigma=1)
        w_prior_shared = pm.Normal("w_prior_shared", mu=0, sigma=1)

        w_strength_initial_opposite = pm.Normal("w_strength_initial_opposite", mu=0, sigma=1)
        w_strength_initial_same = pm.Normal("w_strength_initial_same", mu=0, sigma=1)
        w_strength_initial_nothing = pm.Normal("w_strength_initial_nothing", mu=0, sigma=1)

        w_strength_initial_same_shown_interaction = pm.Normal("w_strength_initial_same_shown_interaction", mu=0, sigma=1)
        w_strength_initial_opposite_shown_interaction = pm.Normal("w_strength_initial_opposite_shown_interaction", mu=0, sigma=1)

        w_strength_COM_opposite = pm.Normal("w_strength_COM_opposite", mu=0, sigma=1)
        w_strength_COM_same = pm.Normal("w_strength_COM_same", mu=0, sigma=1)
        w_strength_COM_nothing = pm.Normal("w_strength_COM_nothing", mu=0, sigma=1)

        w_strength_final_opposite = pm.Normal("w_strength_final_opposite", mu=0, sigma=1)
        w_strength_final_same = pm.Normal("w_strength_final_same", mu=0, sigma=1)
        w_strength_final_nothing = pm.Normal("w_strength_final_nothing", mu=0, sigma=1)

        advice_strength_initial_condition = pm.math.switch(
            pm.math.eq(advice_type, 0),
            w_strength_initial_opposite + w_strength_initial_opposite_shown_interaction * shown_flag,
            pm.math.switch(
                pm.math.eq(advice_type, 1),
                w_strength_initial_same + w_strength_initial_same_shown_interaction * shown_flag,
                w_strength_initial_nothing
            )
        )

        advice_strength_final_condition = pm.math.switch(
            pm.math.eq(advice_type, 0),
            pm.math.switch(
                pm.math.eq(display_type, 0),
                w_strength_final_opposite,
                w_strength_final_opposite
            ),
            pm.math.switch(
                pm.math.eq(advice_type, 1),
                pm.math.switch(
                    pm.math.eq(display_type, 0),
                    w_strength_final_same,
                    w_strength_final_same
                ),
                pm.math.switch(
                    pm.math.eq(display_type, 0),
                    w_strength_final_nothing,
                    w_strength_final_nothing
                )
            )
        )

        advice_strength_switching_condition = pm.math.switch(
            pm.math.eq(advice_type, 0),
            pm.math.switch(
                pm.math.eq(display_type, 0),
                w_strength_COM_opposite,
                w_strength_COM_opposite
            ),
            pm.math.switch(
                pm.math.eq(advice_type, 1),
                pm.math.switch(
                    pm.math.eq(display_type, 0),
                    w_strength_COM_same,
                    w_strength_COM_same
                ),
                pm.math.switch(
                    pm.math.eq(display_type, 0),
                    w_strength_COM_nothing,
                    w_strength_COM_nothing
                )
            )
        )

        advice_strength_initial = advice_strength_initial_condition
        advice_strength_final = advice_strength_final_condition
        advice_strength_switching = advice_strength_switching_condition

        L = (
            intercept_switch +
            w_prior_shared * prior_conf_effective_initial +
            advice_strength_switching * effective_advice_prob_initial_rescaled * advice_direction +
            w_shown_shared * shown_flag
        )

        L_final = (
            intercept_final_conf +
            w_prior_shared * prior_conf_effective_final +
            advice_strength_final * effective_advice_prob_final_rescaled * advice_direction_final +
            w_shown_shared * shown_flag
        )

        L_initial = (
            intercept_initial_conf +
            w_prior_shared * prior_conf_effective_initial +
            advice_strength_initial * effective_advice_prob_initial_rescaled * advice_direction +
            w_shown_shared * shown_flag
        )

        p_switch = pm.Deterministic("p_switch", pm.math.sigmoid(-L))
        switch_obs = pm.Bernoulli("switch_obs", p=p_switch, observed=switch_obs_data)

        posterior_mu_final = pm.Deterministic("posterior_mu_final", pm.math.sigmoid(L_final))
        phi_final = pm.Exponential("phi_final", 1.0)
        alpha_final = pm.Deterministic("alpha_final", posterior_mu_final * phi_final)
        beta_final = pm.Deterministic("beta_final", (1 - posterior_mu_final) * phi_final)

        final_confidence_final_chosen_obs = pm.Beta(
            "final_confidence_final_chosen_obs",
            alpha=alpha_final,
            beta=beta_final,
            observed=final_confidence_final_chosen_obs_data
        )

        posterior_mu_initial = pm.Deterministic("posterior_mu_initial", pm.math.sigmoid(L_initial))
        phi_initial = pm.Exponential("phi_initial", 1.0)
        alpha_initial = pm.Deterministic("alpha_initial", posterior_mu_initial * phi_initial)
        beta_initial = pm.Deterministic("beta_initial", (1 - posterior_mu_initial) * phi_initial)

        final_confidence_initial_chosen_obs = pm.Beta(
            "final_confidence_initial_chosen_obs",
            alpha=alpha_initial,
            beta=beta_initial,
            observed=final_confidence_initial_chosen_obs_data
        )

    return model

🐣

In [ ]:
model = build_latent_model_priorsplit(df, n_conditions = len(condition_labels))
with model:
    trace_latent = pm.sample(draws=500, tune=500, target_accept=0.9, chains=2, progressbar=True, return_inferencedata=True, idata_kwargs={"log_likelihood": True})
    ppc = pm.sample_posterior_predictive(trace_latent, var_names=["switch_obs", "final_confidence_final_chosen_obs", "final_confidence_initial_chosen_obs"])



In [ ]:
def posterior_summary_table(trace, param_names, hdi_prob=0.95):
    summary = az.summary(trace, var_names=param_names, hdi_prob=hdi_prob)
    summary_df = summary[['mean', 'hdi_2.5%', 'hdi_97.5%']]
    return summary_df

params_to_plot = [
    "intercept_switch",
    "intercept_initial_conf",
    "intercept_final_conf",
    "w_strength_initial_opposite",
    "w_strength_initial_opposite_shown_interaction",
    "w_strength_initial_same",
    "w_strength_initial_same_shown_interaction",
    "w_strength_initial_nothing",
    "w_strength_final_opposite",
    "w_strength_final_same",
    "w_strength_final_nothing",
    "w_strength_COM_opposite",
    "w_strength_COM_same",
    "w_strength_COM_nothing",
    "w_prior_shared",
    "w_shown_shared",
]

summary_table = posterior_summary_table(trace_latent, params_to_plot)
summary_table.columns = ['Mean', 'HDI low', 'HDI high']

posterior = trace_latent.posterior
shown_opposite = (posterior["w_strength_initial_opposite"] + posterior["w_strength_initial_opposite_shown_interaction"])
hidden_opposite = posterior["w_strength_initial_opposite"]
shown_same = (posterior["w_strength_initial_same"] + posterior["w_strength_initial_same_shown_interaction"])
hidden_same = posterior["w_strength_initial_same"]

print(summary_table)
print(f"Answer Shown - Opposite: mean={float(shown_opposite.mean()):.3f}, HDI={az.hdi(shown_opposite.values.flatten())}")
print(f"Answer Hidden - Opposite: mean={float(hidden_opposite.mean()):.3f}, HDI={az.hdi(hidden_opposite.values.flatten())}")
print(f"Answer Shown - Same: mean={float(shown_same.mean()):.3f}, HDI={az.hdi(shown_same.values.flatten())}")
print(f"Answer Hidden - Same: mean={float(hidden_same.mean()):.3f}, HDI={az.hdi(hidden_same.values.flatten())}")